# PerfDebug — which records cost too much

Data skew is visible in Spark's own metrics. *Computation* skew is not: a few records
can be far more expensive to process, and no per-task metric says which.

In [ ]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/examples/data"
sys.path.insert(0, f"{ROOT}/python")

## The data

Twelve orders across three customers. One of them, `o8`, is an outlier at
`99999` — every notebook here uses it as the thing to find.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import bigasterisk

spark = (bigasterisk.configure(SparkSession.builder)
    .master("local[2]")
    .appName("perfdebug-notebook")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

orders = spark.read.schema("oid STRING, cid STRING, amount INT").csv(f"{DATA}/orders.txt")
customers = spark.read.schema("cid STRING, name STRING").csv(f"{DATA}/customers.txt")
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

orders.show()

## A pipeline with computation skew

The profiling point goes **above** the expensive step: within a fused pipeline the interval before a record covers the work done for the previous one.

In [ ]:
import time
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

@udf(returnType=IntegerType())
def slow_for_outlier(amount):
    if amount is not None and amount > 1000:
        time.sleep(0.06)
    return amount

# no coalesce here: it puts a CoalesceExec in the plan, which provenance capture
# refuses — and `blame` needs capture
costly = orders.withColumn("checked", slow_for_outlier(col("amount")))
profile = bigasterisk.perfdebug(spark).profile(costly, top_k=5)
profile.df.groupBy("cid").sum("amount").collect()

print("%d records, skew %.1fx the mean" % (profile.records, profile.skew))
for rc in profile.slowest[:3]:
    print(" ", rc)

## Which input made a particular result expensive

In [ ]:
totals = profile.df.groupBy("cid").sum("amount")
totals.collect()
for rc in profile.blame(totals, "cid = 'c2'"):
    print(rc)

## Check

A batched Python UDF blurs per-record attribution, so this checks the totals rather than the ranking.

In [ ]:
assert profile.records > 0
assert profile.total_nanos > 0
print("OK")